# Pseudo-Label Generation — Pipeline A

Generate binary puncta pseudo-labels for segmentation training.
Detects all bright puncta on foreground regions (dendrites, axons, somas)
using Laplacian of Gaussian blob detection, per channel independently.

**Strategy:** Detect puncta everywhere except dark background. No neurite
filtering — any bright punctum on any cellular structure counts.

**Notebook structure:**
1. Configuration
2. Imports & setup
3. Helper functions (LoG detection, mask generation)
4. Visual validation on sample images
5. Batch generation for all images
6. Dataset statistics & summary

## 1. Configuration

All parameters live here. Change these cells, nothing else.

In [ ]:
from types import SimpleNamespace

In [ ]:
cfg = SimpleNamespace(
    seed=42,
)

In [ ]:
data_cfg = SimpleNamespace(
    patch_root="../../data/patches_128",          # directory with .npy patches + index.csv
    output_root="../../data/pseudolabels_A_128",  # output directory for pseudo-label masks
    exclude_patterns=["KONTROLA"],
)

In [ ]:
detect_cfg = SimpleNamespace(
    # LoG blob detection parameters.
    # sigma calibrated to puncta diameter 2-5 px at 107 nm/px.
    # Relationship: blob radius ≈ sqrt(2) * sigma (scikit-image docs).
    # So sigma = radius / sqrt(2) = (diameter/2) / sqrt(2).
    #   diameter 2 px → sigma ≈ 0.71
    #   diameter 5 px → sigma ≈ 1.77
    min_sigma=0.7,
    max_sigma=1.8,
    num_sigma=5,
    # Absolute threshold on LoG response. Images are normalized to [0,1].
    # Start low (permissive), increase if too many false positives on background.
    log_threshold=0.01,
    overlap=0.5,              # suppress overlapping detections
    exclude_border=5,         # ignore blobs within 5px of full-image border
    # Foreground threshold: a pixel is foreground if max(channels) exceeds this.
    # After percentile normalization to [0,1], 0.02 filters pure background.
    foreground_threshold=0.02,
    # Channels to run LoG on. None = all channels independently.
    channels=None,
)

## 2. Imports & Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage.feature import blob_log
from skimage.draw import disk
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath("../.."))
from data_utils.reassemble_patches import (
    reassemble_image,
    slice_to_patches,
    list_image_indices,
)

np.random.seed(cfg.seed)

In [ ]:
output_dir = Path(data_cfg.output_root)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

image_indices = list_image_indices(
    data_cfg.patch_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
print(f"Images to process: {len(image_indices)}")

## 3. Helper Functions

In [ ]:
def detect_puncta_log(channel_image, cfg):
    """Detect puncta in a single 2D channel using Laplacian of Gaussian.

    Args:
        channel_image: (H, W) float32 in [0, 1]
        cfg: SimpleNamespace with LoG parameters

    Returns:
        blobs: (N, 3) array of (row, col, sigma)
    """
    blobs = blob_log(
        channel_image,
        min_sigma=cfg.min_sigma,
        max_sigma=cfg.max_sigma,
        num_sigma=cfg.num_sigma,
        threshold=cfg.log_threshold,
        overlap=cfg.overlap,
        exclude_border=cfg.exclude_border,
    )
    return blobs

In [ ]:
def blobs_to_mask(blobs, image_shape):
    """Convert LoG blob detections to a binary mask.

    Each blob is rendered as a filled disk with radius = sqrt(2) * sigma,
    following the scikit-image convention for LoG blob radius.

    Args:
        blobs: (N, 3) array of (row, col, sigma)
        image_shape: (H, W) tuple

    Returns:
        mask: (H, W) uint8 binary mask
    """
    mask = np.zeros(image_shape, dtype=np.uint8)
    for row, col, sigma in blobs:
        # Radius follows the LoG convention: radius ≈ sqrt(2) * sigma
        radius = max(1, int(np.round(np.sqrt(2) * sigma)))
        rr, cc = disk((int(row), int(col)), radius, shape=image_shape)
        mask[rr, cc] = 1
    return mask

In [ ]:
def make_foreground_mask(image, threshold):
    """Create a binary foreground mask from a multichannel image.

    A pixel is foreground if the max intensity across channels exceeds
    the threshold. This excludes pure-dark background while keeping
    all cellular structures (dendrites, axons, somas).

    Args:
        image: (C, H, W) float32 in [0, 1]
        threshold: float, minimum max-channel intensity for foreground

    Returns:
        mask: (H, W) bool
    """
    return np.max(image, axis=0) > threshold

In [ ]:
def generate_pseudolabels_A(image, detect_cfg):
    """Generate Pipeline-A pseudo-labels for one full image.

    Runs LoG per channel, unions all detections, filters by foreground mask.

    Args:
        image: (C, H, W) float32 in [0, 1]
        detect_cfg: SimpleNamespace with detection parameters

    Returns:
        label_mask: (H, W) uint8 binary mask (1 = puncta, 0 = background)
        per_channel_blobs: list of (N_c, 3) arrays, one per channel
        stats: dict with detection counts
    """
    C, H, W = image.shape
    channels = detect_cfg.channels if detect_cfg.channels is not None else list(range(C))

    per_channel_blobs = []
    combined_mask = np.zeros((H, W), dtype=np.uint8)

    for ch in channels:
        blobs = detect_puncta_log(image[ch], detect_cfg)
        per_channel_blobs.append(blobs)
        if len(blobs) > 0:
            ch_mask = blobs_to_mask(blobs, (H, W))
            combined_mask = np.maximum(combined_mask, ch_mask)

    # Filter by foreground
    fg_mask = make_foreground_mask(image, detect_cfg.foreground_threshold)
    label_mask = combined_mask & fg_mask.astype(np.uint8)

    stats = {
        "total_blobs": sum(len(b) for b in per_channel_blobs),
        "per_channel": [len(b) for b in per_channel_blobs],
        "foreground_fraction": fg_mask.mean(),
        "label_fraction": label_mask.mean(),
    }
    return label_mask, per_channel_blobs, stats

## 4. Visual Validation

Inspect detection results on a few sample images before running the full
batch. Check that real puncta are detected and that background noise is
not producing false positives.

In [ ]:
# Pick a sample image for visual inspection
sample_idx = image_indices[0]
full_img, records = reassemble_image(data_cfg.patch_root, sample_idx)
print(f"Image {sample_idx}: shape={full_img.shape}, "
      f"range=[{full_img.min():.3f}, {full_img.max():.3f}]")

In [ ]:
label_mask, per_ch_blobs, stats = generate_pseudolabels_A(full_img, detect_cfg)
print(f"Detection stats: {stats}")

In [ ]:
# Overview: full image with detected puncta overlaid
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Channel composite (max across channels)
composite = np.max(full_img, axis=0)
axes[0].imshow(composite, cmap="gray", vmin=0, vmax=0.5)
axes[0].set_title(f"Max-channel composite")

# Foreground mask
fg = make_foreground_mask(full_img, detect_cfg.foreground_threshold)
axes[1].imshow(fg, cmap="gray")
axes[1].set_title(f"Foreground mask (threshold={detect_cfg.foreground_threshold})")

# Pseudo-labels
axes[2].imshow(composite, cmap="gray", vmin=0, vmax=0.5)
axes[2].imshow(label_mask, cmap="Reds", alpha=0.4 * label_mask)
axes[2].set_title(f"Pseudo-labels ({stats['total_blobs']} puncta)")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in crop for detailed inspection.
# Choose a region with visible puncta (adjust crop_y, crop_x as needed).
crop_y, crop_x = 400, 400
crop_size = 256
sy = slice(crop_y, crop_y + crop_size)
sx = slice(crop_x, crop_x + crop_size)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
ch_names = ["Ch0 (presynaptic)", "Ch1 (postsynaptic)", "Ch2 (third marker)"]
ch_colors = ["Greens", "Magentas", "Blues"]

for c in range(min(3, full_img.shape[0])):
    axes[c].imshow(full_img[c, sy, sx], cmap="gray", vmin=0, vmax=0.5)
    # Overlay detected blobs as circles
    if c < len(per_ch_blobs):
        for row, col, sigma in per_ch_blobs[c]:
            if crop_y <= row < crop_y + crop_size and crop_x <= col < crop_x + crop_size:
                r = np.sqrt(2) * sigma
                circ = Circle((col - crop_x, row - crop_y), r,
                              fill=False, edgecolor="red", linewidth=0.8)
                axes[c].add_patch(circ)
    axes[c].set_title(ch_names[c] if c < len(ch_names) else f"Ch{c}")
    axes[c].axis("off")

axes[3].imshow(label_mask[sy, sx], cmap="gray")
axes[3].set_title("Combined pseudo-label")
axes[3].axis("off")
plt.suptitle(f"Crop ({crop_y}:{crop_y+crop_size}, {crop_x}:{crop_x+crop_size})")
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: slice labels back into patches and visualize a few.
patch_labels = slice_to_patches(label_mask, records)
print(f"Generated {len(patch_labels)} patch labels")

# Show 8 patches with their labels
sample_fnames = list(patch_labels.keys())[:8]
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i, fname in enumerate(sample_fnames):
    patch_img = np.load(Path(data_cfg.patch_root) / fname)  # (C, H, W)
    patch_lbl = patch_labels[fname]                          # (H, W)
    axes[0, i].imshow(np.max(patch_img, axis=0), cmap="gray", vmin=0, vmax=0.5)
    axes[0, i].set_title(f"{fname[:12]}...", fontsize=7)
    axes[0, i].axis("off")
    axes[1, i].imshow(patch_lbl, cmap="gray")
    n_pos = patch_lbl.sum()
    axes[1, i].set_title(f"{n_pos} px", fontsize=7)
    axes[1, i].axis("off")
plt.suptitle("Top: image patches | Bottom: pseudo-label masks")
plt.tight_layout()
plt.show()

## 5. Batch Generation

Process all images: reassemble → detect puncta → slice → save.
Each pseudo-label is saved as a `(1, 128, 128)` float32 `.npy` file,
matching the `SegmentationPatchDataset` format in `finetune_swinunetr_seg.ipynb`.

In [ ]:
import csv

all_stats = []

for img_idx in tqdm(image_indices, desc="Generating pseudo-labels (Pipeline A)"):
    try:
        full_img, records = reassemble_image(
            data_cfg.patch_root, img_idx,
            exclude_patterns=data_cfg.exclude_patterns,
        )
    except ValueError as e:
        print(f"  Skipping image {img_idx}: {e}")
        continue

    label_mask, per_ch_blobs, stats = generate_pseudolabels_A(full_img, detect_cfg)
    stats["image_index"] = img_idx
    stats["source_image"] = records[0]["source_image"]

    # Slice into patches and save
    patch_labels = slice_to_patches(label_mask, records)
    for fname, patch_lbl in patch_labels.items():
        # Save as (1, H, W) float32 to match segmentation dataset expectations
        out = patch_lbl.astype(np.float32)[np.newaxis, ...]  # (1, 128, 128)
        np.save(output_dir / fname, out)

    stats["n_patches"] = len(patch_labels)
    all_stats.append(stats)

print(f"\nDone. Processed {len(all_stats)} images.")
print(f"Total patches saved: {sum(s['n_patches'] for s in all_stats)}")

In [ ]:
# Copy the index.csv from patch_root so the segmentation dataset can find
# matching image-label pairs by filename.
import shutil
src_csv = Path(data_cfg.patch_root) / "index.csv"
dst_csv = output_dir / "index.csv"
shutil.copy2(src_csv, dst_csv)
print(f"Copied index.csv to {dst_csv}")

## 6. Dataset Statistics

In [ ]:
total_blobs = [s["total_blobs"] for s in all_stats]
label_fracs = [s["label_fraction"] for s in all_stats]
fg_fracs = [s["foreground_fraction"] for s in all_stats]

print(f"Puncta per image:     min={min(total_blobs)}, "
      f"median={int(np.median(total_blobs))}, max={max(total_blobs)}")
print(f"Label pixel fraction: min={min(label_fracs):.4f}, "
      f"median={np.median(label_fracs):.4f}, max={max(label_fracs):.4f}")
print(f"Foreground fraction:  min={min(fg_fracs):.4f}, "
      f"median={np.median(fg_fracs):.4f}, max={max(fg_fracs):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(total_blobs, bins=20, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Puncta per image")
axes[0].set_ylabel("Count")
axes[0].set_title("Puncta count distribution")

axes[1].hist(label_fracs, bins=20, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Fraction of pixels labeled")
axes[1].set_title("Label sparsity")

# Per-channel breakdown
for ch_idx in range(3):
    ch_counts = [s["per_channel"][ch_idx] for s in all_stats if len(s["per_channel"]) > ch_idx]
    axes[2].hist(ch_counts, bins=20, alpha=0.5, label=f"Ch{ch_idx}")
axes[2].set_xlabel("Puncta per image")
axes[2].set_title("Per-channel puncta counts")
axes[2].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save detection statistics for reproducibility
stats_path = output_dir / "detection_stats.csv"
fieldnames = ["image_index", "source_image", "total_blobs",
              "foreground_fraction", "label_fraction", "n_patches"]
with open(stats_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stats)
print(f"Stats saved to {stats_path}")

In [ ]:
# Save the detection config for reproducibility
import json

config_path = output_dir / "detect_config.json"
with open(config_path, "w") as f:
    json.dump(vars(detect_cfg), f, indent=2)
print(f"Config saved to {config_path}")

## Next Steps

1. **Visual inspection**: Scroll through the zoomed crops above. Are real
   puncta detected? Are there false positives on background noise?
2. **Tune `log_threshold`**: If too many FP, increase. If missing puncta, decrease.
3. **Tune `foreground_threshold`**: If too much background passes, increase.
4. **Train segmentation**: Use these pseudo-labels with `finetune_swinunetr_seg.ipynb`:
   ```
   data_cfg.label_root = "../../data/pseudolabels_A_128"
   ```